# EDA 01 — Primeira passagem pelo dataset SRAG

## Objetivo

Realizar a primeira inspeção reproduzível do Parquet oficial do projeto, carregando somente as colunas necessárias.

Esta etapa verifica:

- acesso ao Parquet distribuído pela GitHub Release;
- versão e localização do arquivo;
- schema e colunas disponíveis;
- compatibilidade da lista de colunas da EDA;
- dimensões e consumo de memória;
- tipos de dados;
- período temporal;
- valores ausentes;
- cardinalidade;
- categorias observadas;
- possíveis duplicidades;
- possíveis variáveis posteriores à admissão hospitalar.

## Limites desta etapa

Este notebook não:

- remove registros;
- imputa valores;
- converte ausência em “não”;
- define a lista final de features;
- cria o alvo definitivo;
- realiza split de treino, validação e holdout;
- treina modelos;
- altera configurações ou gates.

As conclusões desta EDA são descritivas e não causais.


In [1]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    """
    Localiza a raiz do projeto procurando pelas pastas src e configs.
    """
    pasta_atual = Path.cwd().resolve()

    for pasta in [pasta_atual, *pasta_atual.parents]:
        possui_src = (pasta / "src").is_dir()
        possui_configs = (pasta / "configs").is_dir()

        if possui_src and possui_configs:
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src.utils import config as C
from src.utils import env
from src.utils import storage


env.carregar_dotenv()

print(f"Raiz do projeto: {ROOT_DIR}")
print(f"Python: {sys.version.split()[0]}")
print(f"Sistema: {platform.system()} {platform.release()}")
print(f"pandas: {pd.__version__}")

Raiz do projeto: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Python: 3.14.7
Sistema: Windows 11
pandas: 3.0.6


In [2]:
print(C.resumo())

print(f"Release configurada: {C.DATA_VERSION}")
print(f"Nome do arquivo: {C.PARQUET_NAME}")
print(f"Diretório de tabelas: {C.TABLES_DIR}")
print(f"Diretório de figuras: {C.FIGURES_DIR}")

ROOT_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
DATA_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data
RAW_DIR           = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed
TREINO_DIR        = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\confi

In [3]:
COLUNAS_ESTRUTURA = [
    "nu_notific",
    "dt_notific",
    "dt_sin_pri",
    "sem_pri",
    "hospital",
    "dt_interna",
]

COLUNAS_CANDIDATAS = [
    "cs_sexo",
    "nu_idade_n",
    "tp_idade",
    "cs_gestant",
    "ave_suino",
    "febre",
    "tosse",
    "garganta",
    "dispneia",
    "diarreia",
    "vomito",
    "outro_sin",
    "puerpera",
    "cardiopati",
    "hematologi",
    "sind_down",
    "hepatica",
    "asma",
    "diabetes",
    "neurologic",
    "pneumopati",
    "imunodepre",
    "renal",
    "obesidade",
    "obes_imc",
    "out_morbi",
    "tabag",
    "vacina",
    "dt_ut_dose",
    "mae_vac",
    "dt_vac_mae",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
    "out_anim",
    "dor_abd",
    "fadiga",
    "perd_olft",
    "perd_pala",
    "vacina_cov",
    "dose_1_cov",
    "dose_2_cov",
    "dose_ref",
    "dose_2ref",
    "dose_adic",
    "dos_re_bi",
]

COLUNAS_AUDITORIA = [
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

COLUNAS_EDA = list(
    dict.fromkeys(
        COLUNAS_ESTRUTURA
        + COLUNAS_CANDIDATAS
        + COLUNAS_AUDITORIA
    )
)

print(f"Colunas de estrutura: {len(COLUNAS_ESTRUTURA)}")
print(f"Colunas candidatas: {len(COLUNAS_CANDIDATAS)}")
print(f"Colunas de auditoria: {len(COLUNAS_AUDITORIA)}")
print(f"Total sem duplicidades: {len(COLUNAS_EDA)}")

Colunas de estrutura: 6
Colunas candidatas: 46
Colunas de auditoria: 4
Total sem duplicidades: 56


In [4]:
caminho_parquet = storage.get_parquet(force=False)

print(f"Arquivo: {caminho_parquet}")
print(f"Existe: {caminho_parquet.exists()}")
print(
    "Tamanho do arquivo: "
    f"{caminho_parquet.stat().st_size / 1024**2:,.2f} MB"
)

Arquivo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed\dados-v1\srag.parquet
Existe: True
Tamanho do arquivo: 236.40 MB


In [5]:
arquivo_parquet = pq.ParquetFile(caminho_parquet)

schema = arquivo_parquet.schema_arrow
colunas_disponiveis = schema.names

metadados_parquet = {
    "arquivo": caminho_parquet.name,
    "versao_dados": C.DATA_VERSION,
    "numero_linhas": arquivo_parquet.metadata.num_rows,
    "numero_colunas": arquivo_parquet.metadata.num_columns,
    "numero_row_groups": arquivo_parquet.metadata.num_row_groups,
}

display(
    pd.DataFrame(
        [metadados_parquet]
    )
)

,arquivo,versao_dados,numero_linhas,numero_colunas,numero_row_groups
0,srag.parquet,dados-v1,4445191,194,9


In [6]:
schema_df = pd.DataFrame(
    {
        "coluna": schema.names,
        "tipo_arrow": [
            str(campo.type)
            for campo in schema
        ],
    }
)

display(schema_df.head(20))
print(f"Total de campos no schema: {len(schema_df)}")

,coluna,tipo_arrow
0,nu_notific,large_string
1,dt_notific,large_string
2,sem_not,large_string
3,dt_sin_pri,large_string
4,sem_pri,large_string
5,sg_uf_not,large_string
6,id_regiona,large_string
7,co_regiona,large_string
8,id_municip,large_string
9,co_mun_not,large_string


Total de campos no schema: 194


In [7]:
conjunto_colunas_disponiveis = set(colunas_disponiveis)

status_colunas = pd.DataFrame(
    {
        "coluna": COLUNAS_EDA,
        "grupo": (
            ["estrutura"] * len(COLUNAS_ESTRUTURA)
            + ["candidata"] * len(COLUNAS_CANDIDATAS)
            + ["auditoria"] * len(COLUNAS_AUDITORIA)
        ),
    }
)

status_colunas["existe_no_parquet"] = (
    status_colunas["coluna"]
    .isin(conjunto_colunas_disponiveis)
)

display(status_colunas)

colunas_ausentes = status_colunas.loc[
    ~status_colunas["existe_no_parquet"],
    "coluna",
].tolist()

if colunas_ausentes:
    print("Colunas ausentes:")
    for coluna in colunas_ausentes:
        print(f"- {coluna}")
else:
    print("Todas as colunas solicitadas existem no Parquet.")

,coluna,grupo,existe_no_parquet
0,nu_notific,estrutura,True
1,dt_notific,estrutura,True
2,dt_sin_pri,estrutura,True
3,sem_pri,estrutura,True
4,hospital,estrutura,True
5,dt_interna,estrutura,True
6,cs_sexo,candidata,True
7,nu_idade_n,candidata,True
8,tp_idade,candidata,True
9,cs_gestant,candidata,True


Todas as colunas solicitadas existem no Parquet.


In [9]:
if colunas_ausentes:
    raise ValueError(
        "A leitura foi interrompida porque existem "
        "colunas ausentes no Parquet: "
        f"{colunas_ausentes}"
    )

print("Validação concluída. A leitura pode continuar.")

Validação concluída. A leitura pode continuar.


In [10]:
df = storage.read_srag(
    columns=COLUNAS_EDA,
)

print(f"Linhas carregadas: {df.shape[0]:,}")
print(f"Colunas carregadas: {df.shape[1]:,}")

display(df.head())

Linhas carregadas: 4,445,191
Colunas carregadas: 56


,nu_notific,dt_notific,dt_sin_pri,sem_pri,hospital,dt_interna,cs_sexo,nu_idade_n,tp_idade,cs_gestant,...,dose_1_cov,dose_2_cov,dose_ref,dose_2ref,dose_adic,dos_re_bi,evolucao,dt_evoluca,dt_encerra,dt_digita
0,315478195042,2018-12-30,2019-01-06,2,1.0,2019-01-06,M,30,3,6,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,2019-01-25,2019-02-15,2019-03-14
1,315478195276,2018-12-30,2019-01-01,1,1.0,2019-01-01,F,7,2,6,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,2019-03-05,2019-01-21,<NA>
2,315478207219,2018-12-30,2018-12-31,1,1.0,2019-01-02,M,1,3,6,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,2019-02-12,2019-05-31,<NA>
3,315478211086,2018-12-30,2019-01-07,2,1.0,2019-01-10,F,5,2,6,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,2019-01-29,2019-02-04,<NA>
4,315478212765,2018-12-30,2019-01-06,2,1.0,2019-01-13,F,3,2,6,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2.0,2019-01-13,2019-05-21,<NA>


In [11]:
if df.empty:
    raise ValueError(
        "O DataFrame foi carregado sem registros."
    )

if list(df.columns) != COLUNAS_EDA:
    print(
        "Atenção: a ordem das colunas carregadas "
        "difere da lista solicitada."
    )

print(f"Dimensão: {df.shape}")
print(f"Índice único: {df.index.is_unique}")
print(f"Quantidade de linhas: {len(df):,}")
print(f"Quantidade de colunas: {df.shape[1]:,}")

Dimensão: (4445191, 56)
Índice único: True
Quantidade de linhas: 4,445,191
Quantidade de colunas: 56


In [12]:
tipos_memoria = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo_pandas": [
            str(df[coluna].dtype)
            for coluna in df.columns
        ],
        "memoria_mb": [
            df[coluna].memory_usage(deep=True) / 1024**2
            for coluna in df.columns
        ],
    }
)

tipos_memoria = tipos_memoria.sort_values(
    "memoria_mb",
    ascending=False,
).reset_index(drop=True)

memoria_total_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Memória total: {memoria_total_mb:,.2f} MB")
display(tipos_memoria)

Memória total: 2,483.20 MB


,coluna,tipo_pandas,memoria_mb
0,nu_notific,string,87.624321
1,dt_notific,string,76.306887
2,dt_sin_pri,string,76.306887
3,dt_digita,string,76.265653
4,dt_interna,string,74.397500
5,dt_encerra,string,73.948575
6,dt_evoluca,string,71.059276
7,hospital,string,46.835349
8,evolucao,string,46.261359
9,tosse,string,45.715399


In [13]:
COLUNAS_DATA = [
    "dt_notific",
    "dt_sin_pri",
    "dt_interna",
    "dt_ut_dose",
    "dt_vac_mae",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

COLUNAS_DATA_EXISTENTES = [
    coluna
    for coluna in COLUNAS_DATA
    if coluna in df.columns
]

datas_convertidas = pd.DataFrame(
    {
        coluna: pd.to_datetime(
            df[coluna],
            errors="coerce",
        )
        for coluna in COLUNAS_DATA_EXISTENTES
    },
    index=df.index,
)

resumo_datas = pd.DataFrame(
    {
        "coluna": COLUNAS_DATA_EXISTENTES,
        "data_minima": [
            datas_convertidas[coluna].min()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "data_maxima": [
            datas_convertidas[coluna].max()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "validas": [
            datas_convertidas[coluna].notna().sum()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "invalidas_ou_ausentes": [
            datas_convertidas[coluna].isna().sum()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
    }
)

display(resumo_datas)

,coluna,data_minima,data_maxima,validas,invalidas_ou_ausentes
0,dt_notific,2017-12-31,2025-12-28,4445191,0
1,dt_sin_pri,2018-12-30,2026-01-03,4445191,0
2,dt_interna,2009-09-04,2224-08-10,4135245,309946
3,dt_ut_dose,2009-03-02,2026-01-29,342997,4102194
4,dt_vac_mae,2000-05-09,2025-12-10,5104,4440087
5,dt_doseuni,2013-04-25,2026-01-27,8863,4436328
6,dt_1_dose,2012-08-06,2026-01-29,9001,4436190
7,dt_2_dose,2012-09-14,2026-02-06,6098,4439093
8,dt_evoluca,2019-01-01,2026-02-26,3839374,605817
9,dt_encerra,2019-01-05,2026-02-28,4142339,302852


In [14]:
if "dt_notific" not in datas_convertidas.columns:
    raise KeyError(
        "A coluna dt_notific não está disponível "
        "para avaliar o período temporal."
    )

ano_notificacao = (
    datas_convertidas["dt_notific"]
    .dt.year
    .rename("ano_notificacao")
)

volume_por_ano = (
    ano_notificacao
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ano_notificacao")
    .reset_index(name="quantidade_registros")
)

volume_por_ano["percentual"] = (
    volume_por_ano["quantidade_registros"]
    / len(df)
    * 100
)

display(volume_por_ano)

,ano_notificacao,quantidade_registros,percentual
0,2017,3,0.000067
1,2018,48637,1.094149
2,2019,1148795,25.843546
3,2020,1773883,39.905664
4,2021,582657,13.107581
5,2023,552902,12.438206
6,2024,334220,7.518687
7,2025,4094,0.092100


In [15]:
memoria_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Memória total utilizada: {memoria_mb:,.2f} MB")

Memória total utilizada: 2,483.20 MB


In [25]:
memoria_colunas = (
    df.memory_usage(deep=True)
    .div(1024**2)
    .rename("memoria_mb")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "coluna"})
)

display(memoria_colunas)

,coluna,memoria_mb
0,nu_notific,87.624195
1,dt_sin_pri,76.306761
2,dt_notific,76.306761
3,dt_digita,76.265527
4,dt_interna,74.397374
5,dt_encerra,73.948449
6,dt_evoluca,71.059150
7,hospital,46.835223
8,evolucao,46.261233
9,tosse,45.715273


In [16]:
ausencias = pd.DataFrame(
    {
        "coluna": df.columns,
        "quantidade_ausente": [
            df[coluna].isna().sum()
            for coluna in df.columns
        ],
        "quantidade_preenchida": [
            df[coluna].notna().sum()
            for coluna in df.columns
        ],
    }
)

ausencias["percentual_ausente"] = (
    ausencias["quantidade_ausente"]
    / len(df)
    * 100
)

ausencias["percentual_preenchido"] = (
    ausencias["quantidade_preenchida"]
    / len(df)
    * 100
)

ausencias = ausencias.sort_values(
    "percentual_ausente",
    ascending=False,
).reset_index(drop=True)

display(ausencias)

,coluna,quantidade_ausente,quantidade_preenchida,percentual_ausente,percentual_preenchido
0,tabag,4445187,4,99.999910,0.000090
1,out_anim,4440432,4759,99.892940,0.107060
2,dt_vac_mae,4440084,5107,99.885112,0.114888
3,dt_2_dose,4439090,6101,99.862751,0.137249
4,dt_doseuni,4436325,8866,99.800549,0.199451
5,dt_1_dose,4436187,9004,99.797444,0.202556
6,dose_adic,4427283,17908,99.597138,0.402862
7,obes_imc,4391362,53829,98.789051,1.210949
8,dos_re_bi,4357754,87437,98.032998,1.967002
9,mae_vac,4349582,95609,97.849159,2.150841


In [17]:
df_temporal = df.copy()

df_temporal["ano_notificacao"] = ano_notificacao

preenchimento_por_ano = (
    df_temporal
    .groupby("ano_notificacao", dropna=False)
    [COLUNAS_EDA]
    .apply(lambda grupo: grupo.notna().mean() * 100)
    .reset_index()
)

display(preenchimento_por_ano)

,ano_notificacao,nu_notific,dt_notific,dt_sin_pri,sem_pri,hospital,dt_interna,cs_sexo,nu_idade_n,tp_idade,...,dose_1_cov,dose_2_cov,dose_ref,dose_2ref,dose_adic,dos_re_bi,evolucao,dt_evoluca,dt_encerra,dt_digita
0,2017,100.0,100.0,100.0,100.0,100.000000,100.000000,100.0,100.0,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,100.000000,66.666667
1,2018,100.0,100.0,100.0,100.0,99.438699,97.316858,100.0,100.0,100.0,...,0.096634,0.071962,0.024673,0.008224,0.002056,0.006168,94.189609,84.080021,96.418365,65.421387
2,2019,100.0,100.0,100.0,100.0,97.624467,92.570650,100.0,100.0,100.0,...,2.152952,2.026471,1.185677,0.472321,0.036560,0.106198,93.641076,87.272316,94.055597,100.000000
3,2020,100.0,100.0,100.0,100.0,97.372600,93.659616,100.0,100.0,100.0,...,20.068065,13.423320,1.929891,0.467393,0.029652,0.087435,91.926751,85.794610,92.464497,99.999887
4,2021,100.0,100.0,100.0,100.0,96.706467,92.862868,100.0,100.0,100.0,...,56.679144,52.851163,27.512585,4.705170,0.077576,0.151547,91.922006,85.642496,91.870689,100.000000
5,2023,100.0,100.0,100.0,100.0,97.518909,94.275116,100.0,100.0,100.0,...,48.126069,43.851894,25.623709,14.890704,1.850961,8.463525,94.711178,88.019938,94.218686,100.000000
6,2024,100.0,100.0,100.0,100.0,97.919335,94.476991,100.0,100.0,100.0,...,44.641853,40.562504,22.127042,12.935492,1.848782,10.914368,94.387230,85.392257,94.256478,100.000000
7,2025,100.0,100.0,100.0,100.0,97.044455,89.106009,100.0,100.0,100.0,...,51.172447,47.557401,25.207621,14.582316,2.344895,12.383977,84.807035,71.787982,84.831461,100.000000


In [18]:
cardinalidade = pd.DataFrame(
    {
        "coluna": df.columns,
        "valores_distintos_sem_na": [
            df[coluna].nunique(dropna=True)
            for coluna in df.columns
        ],
        "valores_distintos_com_na": [
            df[coluna].nunique(dropna=False)
            for coluna in df.columns
        ],
    }
)

cardinalidade = cardinalidade.sort_values(
    "valores_distintos_com_na",
    ascending=False,
).reset_index(drop=True)

display(cardinalidade)

,coluna,valores_distintos_sem_na,valores_distintos_com_na
0,nu_notific,4445169,4445169
1,dt_interna,3920,3921
2,dt_ut_dose,2838,2839
3,dt_evoluca,2616,2617
4,dt_encerra,2611,2612
5,dt_sin_pri,2562,2562
6,dt_digita,2554,2555
7,dose_1_cov,1896,1897
8,dose_2_cov,1818,1819
9,dt_vac_mae,1579,1580


In [19]:
LIMITE_CARDINALIDADE = 30

colunas_categoricas_baixas = cardinalidade.loc[
    cardinalidade["valores_distintos_com_na"]
    <= LIMITE_CARDINALIDADE,
    "coluna",
].tolist()

frequencias_categoricas = []

for coluna in colunas_categoricas_baixas:
    frequencia = (
        df[coluna]
        .value_counts(
            dropna=False,
            normalize=False,
        )
        .rename("quantidade")
        .reset_index()
        .rename(columns={"index": "valor"})
    )

    frequencia["coluna"] = coluna
    frequencia["percentual"] = (
        frequencia["quantidade"]
        / len(df)
        * 100
    )

    frequencias_categoricas.append(frequencia)

frequencias_categoricas_df = pd.concat(
    frequencias_categoricas,
    ignore_index=True,
)

display(frequencias_categoricas_df.head(100))

,vacina,quantidade,coluna,percentual,sind_down,dt_notific,hematologi,diabetes,pneumopati,neurologic,...,diarreia,vomito,dispneia,perd_pala,febre,outro_sin,vacina_cov,cs_sexo,tp_idade,tabag
0,<NA>,1184651,vacina,26.650171,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2.0,903812,vacina,20.332355,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,9,664537,vacina,14.949571,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,9.0,641839,vacina,14.438952,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2,546575,vacina,12.295872,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,<NA>,22450,puerpera,0.50504,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
96,<NA>,11230,puerpera,0.252633,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
97,<NA>,9048,puerpera,0.203546,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
98,<NA>,3593,puerpera,0.080829,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [20]:
if "nu_notific" in df.columns:
    identificadores_preenchidos = df["nu_notific"].notna().sum()

    identificadores_duplicados = (
        df.loc[
            df["nu_notific"].notna(),
            "nu_notific",
        ]
        .duplicated(keep=False)
        .sum()
    )

    resumo_identificador = pd.DataFrame(
        {
            "metrica": [
                "linhas_totais",
                "identificadores_preenchidos",
                "linhas_com_identificador_duplicado",
            ],
            "valor": [
                len(df),
                identificadores_preenchidos,
                identificadores_duplicados,
            ],
        }
    )

    display(resumo_identificador)
else:
    print(
        "nu_notific não foi carregada. "
        "A auditoria de identificador não será executada."
    )

,metrica,valor
0,linhas_totais,4445191
1,identificadores_preenchidos,4445191
2,linhas_com_identificador_duplicado,44


In [21]:
COLUNAS_POTENCIALMENTE_POS_ADMISSAO = [
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

auditoria_vazamento = pd.DataFrame(
    {
        "coluna": COLUNAS_POTENCIALMENTE_POS_ADMISSAO,
        "existe_no_dataframe": [
            coluna in df.columns
            for coluna in COLUNAS_POTENCIALMENTE_POS_ADMISSAO
        ],
        "uso_nesta_eda": "auditoria",
        "candidata_feature": False,
        "justificativa": (
            "Desfecho ou informação potencialmente "
            "posterior à admissão hospitalar."
        ),
    }
)

display(auditoria_vazamento)

,coluna,existe_no_dataframe,uso_nesta_eda,candidata_feature,justificativa
0,evolucao,True,auditoria,False,Desfecho ou informação potencialmente posterio...
1,dt_evoluca,True,auditoria,False,Desfecho ou informação potencialmente posterio...
2,dt_encerra,True,auditoria,False,Desfecho ou informação potencialmente posterio...
3,dt_digita,True,auditoria,False,Desfecho ou informação potencialmente posterio...


In [22]:
resumo_primeira_passagem = pd.DataFrame(
    {
        "metrica": [
            "versao_dados",
            "arquivo",
            "linhas_parquet",
            "colunas_parquet",
            "linhas_carregadas",
            "colunas_carregadas",
            "memoria_mb",
            "data_minima_notificacao",
            "data_maxima_notificacao",
            "colunas_ausentes",
        ],
        "valor": [
            C.DATA_VERSION,
            caminho_parquet.name,
            arquivo_parquet.metadata.num_rows,
            arquivo_parquet.metadata.num_columns,
            df.shape[0],
            df.shape[1],
            round(memoria_total_mb, 2),
            datas_convertidas["dt_notific"].min(),
            datas_convertidas["dt_notific"].max(),
            len(colunas_ausentes),
        ],
    }
)

display(resumo_primeira_passagem)

,metrica,valor
0,versao_dados,dados-v1
1,arquivo,srag.parquet
2,linhas_parquet,4445191
3,colunas_parquet,194
4,linhas_carregadas,4445191
5,colunas_carregadas,56
6,memoria_mb,2483.2
7,data_minima_notificacao,2017-12-31 00:00:00
8,data_maxima_notificacao,2025-12-28 00:00:00
9,colunas_ausentes,0


In [24]:
C.TABLES_DIR.mkdir(
parents=True,
exist_ok=True,
)

print(f"Tabelas serão salvas em: {C.TABLES_DIR}")

Tabelas serão salvas em: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables


In [25]:
tabelas_exportacao = {
    "eda_01_resumo_primeira_passagem.csv": (
        resumo_primeira_passagem
    ),
    "eda_01_status_colunas.csv": (
        status_colunas
    ),
    "eda_01_tipos_memoria.csv": (
        tipos_memoria
    ),
    "eda_01_resumo_datas.csv": (
        resumo_datas
    ),
    "eda_01_volume_por_ano.csv": (
        volume_por_ano
    ),
    "eda_01_ausencias.csv": (
        ausencias
    ),
    "eda_01_preenchimento_por_ano.csv": (
        preenchimento_por_ano
    ),
    "eda_01_cardinalidade.csv": (
        cardinalidade
    ),
    "eda_01_frequencias_categoricas.csv": (
        frequencias_categoricas_df
    ),
    "eda_01_auditoria_vazamento.csv": (
        auditoria_vazamento
    ),
}

for nome_arquivo, tabela in tabelas_exportacao.items():
    caminho_saida = C.TABLES_DIR / nome_arquivo

    tabela.to_csv(
        caminho_saida,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Salvo: {caminho_saida}")

Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_resumo_primeira_passagem.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_status_colunas.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_tipos_memoria.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_resumo_datas.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_volume_por_ano.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_ausencias.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_preenchimento_por_ano.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_cardinalidade.csv
Salvo: C:\Users\olivem85\OneDriv

## Decisões pendentes após a primeira passagem

### Schema

- Confirmar o significado das colunas ausentes ou com nomes diferentes.
- Alinhar os nomes físicos do Parquet aos nomes documentados.
- Verificar especificamente `dt_evoluca` versus `dt_evolucao`.
- Verificar especificamente `dt_encerra` versus `dt_encerramento`.

### Unidade de análise

- Confirmar se `nu_notific` identifica de forma adequada uma notificação.
- Investigar duplicidades antes de definir qualquer remoção.
- Não interpretar registros automaticamente como pessoas únicas.

### Ausências

- Distinguir vazio, ignorado, não se aplica, campo inexistente e resposta negativa.
- Não converter ausência em zero ou “não” sem regra documentada.
- Comparar preenchimento entre anos antes de excluir variáveis.

### Datas

- Investigar datas inválidas e sequências temporais impossíveis.
- Definir o período elegível do estudo.
- Identificar anos incompletos.
- Levar a cobertura temporal para a decisão do Gate G2.

### Features e vazamento

- A presença de uma coluna na EDA não significa aprovação como feature.
- `evolucao`, `dt_evoluca`, `dt_encerra` e `dt_digita` permanecem apenas para auditoria.
- Revisar todas as colunas conforme a disponibilidade plausível até a admissão.
- Atualizar a lista de variáveis proibidas somente após decisão formal.

### Próxima etapa

- Executar uma EDA descritiva por ano e por grupo.
- Comparar frequências, preenchimento e categorias entre períodos.
- Criar um relatório de hipóteses e riscos.
- Não usar o holdout para escolher features, tratamentos ou modelos.

In [26]:
verificacoes_finais = {
    "dataframe_nao_vazio": not df.empty,
    "colunas_solicitadas_presentes": not colunas_ausentes,
    "dt_notific_disponivel": "dt_notific" in df.columns,
    "evolucao_apenas_auditoria": (
        "evolucao" in COLUNAS_AUDITORIA
        and "evolucao" not in COLUNAS_CANDIDATAS
    ),
    "diretorio_tabelas_existe": C.TABLES_DIR.exists(),
}

verificacoes_finais_df = pd.DataFrame(
    {
        "verificacao": verificacoes_finais.keys(),
        "resultado": verificacoes_finais.values(),
    }
)

display(verificacoes_finais_df)

if not all(verificacoes_finais.values()):
    raise RuntimeError(
        "Uma ou mais verificações finais falharam."
    )

print("Primeira passagem concluída com sucesso.")

,verificacao,resultado
0,dataframe_nao_vazio,True
1,colunas_solicitadas_presentes,True
2,dt_notific_disponivel,True
3,evolucao_apenas_auditoria,True
4,diretorio_tabelas_existe,True


Primeira passagem concluída com sucesso.
